# CILLM RAG 工作坊 Notebook

這份 notebook 依照投影片分成三段實作：

1. **實作 A**：測試 3 個 LLM 的回覆與時間，並快速確認 Embedding API
2. **實作 B1**：自己用 LlamaIndex + LangChain 做 10 題單輪 RAG
3. **實作 B2**：用 CILLM GPU Server Vector DB API 做同一批 10 題單輪 RAG，並比較 B1/B2 時間差
4. **實作 C**：再建立另一批 10 題 QA，用 Pydantic structured output 讓 LLM 判斷要查哪個 Vector DB，再接 RAG 回答

> 注意：範例 QA 是教學用虛構資料，不代表華航正式制度。正式上線前要換成核准過的文件內容與 reference。


## 0. 安裝與設定

投影片建議環境：Python 3.11.x、Jupyter Notebook、requests、LlamaIndex、LangChain、Pydantic。

如果你已經安裝過，可以跳過下一格。


In [ ]:
# 如已安裝可跳過
# %pip install requests llama-index langchain langchain-openai pydantic


In [ ]:
import os
import json
import time
import getpass
import statistics
from datetime import datetime
from pprint import pprint
from typing import Any, List, Literal

import requests

BASE_URL = os.getenv("CILLM_BASE_URL", "https://cillmtest.china-airlines.com").rstrip("/")
DEFAULT_CHAT_MODEL = os.getenv("CILLM_CHAT_MODEL", "openai/gpt-oss-120b")
EMBED_MODEL = os.getenv("CILLM_EMBED_MODEL", "nvidia/llama-3.2-nv-embedqa-1b-v2")

DEFAULT_LLM_MODELS = [
    "meta/llama-3.3-70b-instruct",
    "google/gemma-4-31b-it",
    "openai/gpt-oss-120b",
]
LLM_MODELS = [
    item.strip()
    for item in os.getenv("CILLM_LLM_MODELS", ",".join(DEFAULT_LLM_MODELS)).split(",")
    if item.strip()
]

API_KEY = os.getenv("CILLM_API_KEY") or getpass.getpass("請輸入 CILLM_API_KEY：")

HEADERS = {
    "accept": "application/json",
    "Content-Type": "application/json",
    "CILLM_API_KEY": API_KEY,
}

print("BASE_URL:", BASE_URL)
print("LLM_MODELS:", LLM_MODELS)
print("EMBED_MODEL:", EMBED_MODEL)


In [ ]:
def api_request(method, path, body=None, params=None, timeout=120):
    url = f"{BASE_URL}/{path.lstrip('/')}"
    response = requests.request(
        method=method,
        url=url,
        headers=HEADERS,
        json=body,
        params=params,
        timeout=timeout,
    )
    try:
        data = response.json()
    except Exception:
        data = response.text

    if not response.ok:
        print("HTTP", response.status_code, method, path)
        pprint(data)
        response.raise_for_status()

    return data


def timed_call(func, *args, **kwargs):
    started = time.perf_counter()
    value = func(*args, **kwargs)
    elapsed = time.perf_counter() - started
    return value, elapsed


def chat(messages, model=DEFAULT_CHAT_MODEL, temperature=0.2, max_tokens=512, response_format=None, timeout=180):
    body = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False,
    }
    if response_format is not None:
        body["response_format"] = response_format
    result = api_request("POST", "/v1/chat/completions", body, timeout=timeout)
    return result["choices"][0]["message"]["content"]


def embed_texts(texts, model=EMBED_MODEL, input_type="query", timeout=180):
    if isinstance(texts, str):
        texts = [texts]
    body = {
        "model": model,
        "input": texts,
        "encoding_format": "float",
        "input_type": input_type,
        "truncate": "END",
    }
    result = api_request("POST", "/v1/embeddings", body, timeout=timeout)
    return [item["embedding"] for item in result["data"]]


def print_timing_table(rows, title="Timing"):
    print(f"\n{title}")
    print("-" * 96)
    print(f"{'#':>2}  {'label':<34} {'retrieval':>10} {'generation':>11} {'total':>10}")
    print("-" * 96)
    for idx, row in enumerate(rows, 1):
        label = row.get("label") or row.get("question", "")[:34]
        retrieval = row.get("retrieval_seconds")
        generation = row.get("generation_seconds")
        total = row.get("total_seconds", row.get("seconds", 0))
        print(
            f"{idx:>2}  {label[:34]:<34} "
            f"{retrieval if retrieval is not None else 0:>10.2f} "
            f"{generation if generation is not None else 0:>11.2f} "
            f"{total:>10.2f}"
        )
    totals = [row.get("total_seconds", row.get("seconds", 0)) for row in rows]
    if totals:
        print("-" * 96)
        print(f"total={sum(totals):.2f}s, avg={statistics.mean(totals):.2f}s, min={min(totals):.2f}s, max={max(totals):.2f}s")


## 1. 連線檢查：key、scopes、可用模型

這格先確認目前 key 能看到哪些模型。後面的實作只會用到非 RBAC/Guardrail 的 LLM、Embedding、Vector DB API。


In [ ]:
current_key = api_request("GET", "/v1/rbac/keys/current")
scopes = set(current_key.get("scopes", []))
print("目前 key user_id:", current_key.get("user_id"))
print("目前 key status:", current_key.get("status"))
print("\n我的 scopes：")
for scope in sorted(scopes):
    print("-", scope)

available_llms = api_request("GET", "/v1/llms").get("data", [])
available_embedding_models = api_request("GET", "/v1/embedding-models").get("data", [])
available_llm_ids = {model["id"] for model in available_llms}

print("\n目前可用 LLM：")
for model in available_llms:
    print("-", model["id"])

print("\n目前可用 Embedding Models：")
for model in available_embedding_models:
    print("-", model["id"])


## 實作 A1：設計 Prompt，切換不同 LLM

投影片 A1 的目標是先熟悉 Jupyter Notebook 的操作方式，並用同一組 prompt 呼叫 CILLM 支援的不同 LLM。

課堂操作：

- 請到 Jupyter Notebook 頁面
- 欲執行特定 Code Cell，請透過 `Ctrl + Enter`
- 欲執行特定 Code Cell，請透過 `Shift + Enter`
- 請設計一組 Prompt，然後呼叫 CILLM 支援的 LLM
- 請切換不同的 LLM 名字，感受不同 LLM 的表現程度

下面的 `practice_a1_prompt` 可以直接修改。建議先使用同一個 prompt 跑三個模型，再比較「回答內容、口吻、完整度、耗時」。


In [ ]:
# 實作 A1：請修改這一組 Prompt，再用三個 LLM 各跑一次。
practice_a1_prompt = """
請用繁體中文回答：
如果要把 RAG 用在航空公司內部知識查詢，最適合的三個使用情境是什麼？
請用 3 個條列回答，每一點都要包含一個具體工作場景。
""".strip()

# 可直接改這個清單，感受不同 LLM 的表現程度。
practice_a1_models = [
    "meta/llama-3.3-70b-instruct",
    "google/gemma-4-31b-it",
    "openai/gpt-oss-120b",
]

models_to_test = [
    model
    for model in practice_a1_models
    if not available_llm_ids or model in available_llm_ids
]
if len(models_to_test) < len(practice_a1_models):
    missing_models = sorted(set(practice_a1_models) - set(models_to_test))
    print("以下模型目前沒有列在可用清單中，會先略過：", missing_models)

practice_a1_results = []
for model in models_to_test:
    print(f"\n=== {model} ===")
    answer, seconds = timed_call(
        chat,
        [
            {"role": "system", "content": "你是 CILLM 教學助教，回答要精準、清楚、可操作。"},
            {"role": "user", "content": practice_a1_prompt},
        ],
        model=model,
        temperature=0.2,
        max_tokens=600,
    )
    practice_a1_results.append({
        "label": model,
        "seconds": seconds,
        "answer": answer,
    })
    print(f"耗時：{seconds:.2f} 秒")
    print(answer)

## 實作 A2：三組文字的 Cosine Similarity

投影片 A2 的目標是感受 embedding 如何把文字轉成向量，並用 cosine similarity 比較語意接近程度。

課堂操作：

- 請設計三組文字
- 其中兩組在語意上要接近
- 第三組可以毫不相關
- 透過 Cosine Similarity 計算兩兩成對的相似度

範例文字沿用投影片：

- 第一組：中華航空的搭機體驗非常好
- 第二組：我很喜歡搭中華航空的飛機
- 第三組：我想知道波音 747 的特點有哪些


In [ ]:
import math
from itertools import combinations


# 實作 A2：請修改下面三組文字。
# 建議第一組與第二組語意接近，第三組改成比較不相關的主題。
practice_a2_texts = {
    "第一組": "中華航空的搭機體驗非常好",
    "第二組": "我很喜歡搭中華航空的飛機",
    "第三組": "我想知道波音 747 的特點有哪些",
}


def cosine_similarity(vector_a, vector_b):
    dot = sum(a * b for a, b in zip(vector_a, vector_b))
    norm_a = math.sqrt(sum(a * a for a in vector_a))
    norm_b = math.sqrt(sum(b * b for b in vector_b))
    if norm_a == 0 or norm_b == 0:
        return 0
    return dot / (norm_a * norm_b)


labels = list(practice_a2_texts.keys())
texts = [practice_a2_texts[label] for label in labels]

vectors, embedding_seconds = timed_call(
    embed_texts,
    texts,
    model=EMBED_MODEL,
    input_type="query",
)

print(f"Embedding 耗時：{embedding_seconds:.2f} 秒")
print("向量數量：", len(vectors))
print("向量維度：", len(vectors[0]))

vector_by_label = dict(zip(labels, vectors))

print("\n兩兩 Cosine Similarity")
print("-" * 86)
print(f"{'pair':<18} {'similarity':>12}  text A / text B")
print("-" * 86)
for label_a, label_b in combinations(labels, 2):
    score = cosine_similarity(vector_by_label[label_a], vector_by_label[label_b])
    text_a = practice_a2_texts[label_a]
    text_b = practice_a2_texts[label_b]
    print(f"{label_a + ' vs ' + label_b:<18} {score:>12.4f}  {text_a} / {text_b}")

print("\n觀察重點：第一組 vs 第二組通常應該最高，因為語意接近；第三組和前兩組通常較低。")


## 實作 B：同一組 10 題 QA，做兩種單輪 RAG

- **B1**：自己用 LlamaIndex 建本機向量 index，用 LangChain 組 prompt 並呼叫 LLM
- **B2**：用 CILLM GPU Server 的 Vector DB API 建庫、上傳 QA、檢索，再呼叫 LLM

這一節會用同一組 10 題 benchmark 問題，最後比較 B1/B2 的總時間與平均時間。


In [ ]:
from pydantic import BaseModel, Field
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.embeddings import BaseEmbedding
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class CILLMEmbedding(BaseEmbedding):
    """讓 LlamaIndex 使用 CILLM `/v1/embeddings` API。"""

    model_name: str = EMBED_MODEL

    def _get_query_embedding(self, query: str) -> List[float]:
        return embed_texts([query], model=self.model_name, input_type="query")[0]

    async def _aget_query_embedding(self, query: str) -> List[float]:
        return self._get_query_embedding(query)

    def _get_text_embedding(self, text: str) -> List[float]:
        return embed_texts([text], model=self.model_name, input_type="passage")[0]

    def _get_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        return embed_texts(texts, model=self.model_name, input_type="passage")


Settings.embed_model = CILLMEmbedding(model_name=EMBED_MODEL)
Settings.llm = None

CILLM_LANGCHAIN_MODEL = ChatOpenAI(
    model=DEFAULT_CHAT_MODEL,
    base_url=f"{BASE_URL}/v1",
    api_key="not-used-by-cillm",
    default_headers={"CILLM_API_KEY": API_KEY},
    temperature=0,
    timeout=180,
)


def prompt_value_to_cillm_messages(prompt_value):
    """把 LangChain PromptValue 轉成 CILLM chat API 的 messages 格式。"""
    role_map = {"system": "system", "human": "user", "ai": "assistant"}
    return [
        {"role": role_map.get(message.type, "user"), "content": message.content}
        for message in prompt_value.to_messages()
    ]


In [ ]:
pax_qa = [
    {"question": "旅客可以在哪些時間使用線上報到？", "answer": "旅客可於航班起飛前 48 小時至 1 小時間使用線上報到；部分航線仍需依現場規定辦理。", "ref": "CI-PAX-ONLINE-CHECKIN-2026#2.1"},
    {"question": "自助報到機可以列印哪些文件？", "answer": "自助報到機可列印登機證，若該站支援行李託運流程，也可列印行李條。", "ref": "CI-PAX-KIOSK-2026#1.4"},
    {"question": "手提行李超過限制時應如何處理？", "answer": "若手提行李超過尺寸或重量限制，旅客需依地勤人員指示改以託運行李處理。", "ref": "CI-BAG-CABIN-2026#3.2"},
    {"question": "託運行李遺失時旅客應先去哪裡申報？", "answer": "旅客應於抵達機場行李服務櫃檯申報，並取得行李異常報告編號。", "ref": "CI-BAG-IRREG-2026#4.1"},
    {"question": "特殊餐點最晚何時要提出申請？", "answer": "特殊餐點通常需於航班起飛前 24 小時完成申請，實際可供應品項依航線與艙等而定。", "ref": "CI-PAX-MEAL-2026#2.3"},
    {"question": "輪椅服務是否需要事先申請？", "answer": "建議旅客於訂位或起飛前盡早提出輪椅服務需求，以利機場與航班作業安排。", "ref": "CI-PAX-WCHR-2026#1.2"},
    {"question": "登機門關閉後旅客還能登機嗎？", "answer": "登機門關閉後通常不得再登機，旅客需洽地勤人員依誤機或改票流程處理。", "ref": "CI-PAX-BOARDING-2026#5.5"},
    {"question": "嬰兒旅客是否有免費託運行李額度？", "answer": "嬰兒旅客的行李額度依票種與航線規定不同，常見情況可託運一件嬰兒車或安全座椅。", "ref": "CI-BAG-INFANT-2026#2.2"},
    {"question": "寵物託運需要哪些基本條件？", "answer": "寵物託運需事先申請，並符合航空箱、健康文件、目的地檢疫與航班可承載條件。", "ref": "CI-PAX-PET-2026#3.1"},
    {"question": "航班延誤時旅客可在哪裡查詢最新資訊？", "answer": "旅客可透過官網、行動 App、機場顯示螢幕或地勤櫃檯查詢最新航班狀態。", "ref": "CI-PAX-DELAY-2026#1.1"},
]

pax_benchmark_questions = [
    "線上報到最早和最晚可以在什麼時間辦理？",
    "我在自助報到機可以拿到登機證嗎？",
    "手提行李太大或太重時現場會怎麼處理？",
    "抵達後找不到託運行李，第一步要去哪裡？",
    "如果想訂特殊餐，最晚要什麼時候講？",
    "需要輪椅服務是不是要先提出？",
    "登機門已關閉還能不能上飛機？",
    "嬰兒旅客可以託運嬰兒車嗎？",
    "寵物要託運時需要先準備什麼？",
    "航班延誤時我可以在哪裡看最新狀態？",
]

pax_documents = [
    Document(
        text=f"問題：{item['question']}\n答案：{item['answer']}\n來源：{item['ref']}",
        metadata={"ref": item["ref"], "topic": "旅客服務與行李"},
    )
    for item in pax_qa
]

print("第一組 QA 數量：", len(pax_qa))
print("benchmark 問題數量：", len(pax_benchmark_questions))
print(pax_documents[0].text)


### 實作 B1：LlamaIndex + LangChain 自建 10 題 RAG

B1 的資料流：

`QA documents` → `CILLM Embedding API` → `LlamaIndex VectorStoreIndex` → `LlamaIndex retriever` → `LangChain prompt` → `CILLM Chat API`


In [ ]:
pax_index, pax_index_build_seconds = timed_call(
    VectorStoreIndex.from_documents,
    pax_documents,
    embed_model=Settings.embed_model,
)
print(f"本機 LlamaIndex index 建立完成，耗時 {pax_index_build_seconds:.2f} 秒")


def llamaindex_retrieve(question, index=pax_index, top_k=3):
    retriever = index.as_retriever(similarity_top_k=top_k)
    return retriever.retrieve(question)


hits = llamaindex_retrieve("我想知道行李不見怎麼辦？", top_k=3)
for node in hits:
    score = node.score if node.score is not None else 0
    ref = node.node.metadata.get("ref")
    first_line = node.node.text.splitlines()[0]
    print(round(score, 4), ref, first_line)


In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "你是繁體中文 RAG 助教。請根據檢索內容回答，不要編造；回答最後請列出引用來源。",
    ),
    (
        "human",
        """
請根據下方知識庫內容回答使用者問題。
如果知識庫沒有答案，請說明目前資料不足，不要自行編造。

【知識庫】
{context}

【使用者問題】
{question}
""".strip(),
    ),
])


def format_llamaindex_nodes(nodes):
    return "\n\n".join(
        f"[分數 {(node.score or 0):.4f}]\n{node.node.text}"
        for node in nodes
    )


def invoke_langchain_text(prompt_value, model=DEFAULT_CHAT_MODEL):
    # Keep LangChain for prompt assembly; call CILLM REST chat for gateway compatibility.
    return chat(
        prompt_value_to_cillm_messages(prompt_value),
        model=model,
        temperature=0,
        max_tokens=700,
    )


def local_rag_chat(user_question, index=pax_index, top_k=3):
    started = time.perf_counter()
    nodes, retrieval_seconds = timed_call(llamaindex_retrieve, user_question, index=index, top_k=top_k)
    context = format_llamaindex_nodes(nodes)
    prompt_value = rag_prompt.invoke({"context": context, "question": user_question})
    answer, generation_seconds = timed_call(invoke_langchain_text, prompt_value)
    total_seconds = time.perf_counter() - started
    return {
        "question": user_question,
        "answer": answer,
        "refs": [node.node.metadata.get("ref") for node in nodes],
        "retrieval_seconds": retrieval_seconds,
        "generation_seconds": generation_seconds,
        "total_seconds": total_seconds,
    }


practice_b1_results = []
for question in pax_benchmark_questions:
    result = local_rag_chat(question, top_k=3)
    practice_b1_results.append(result)
    print("\nQ:", question)
    print("refs:", result["refs"])
    print("seconds:", round(result["total_seconds"], 2))
    print(result["answer"])

print_timing_table(practice_b1_results, title="實作 B1：LlamaIndex + LangChain 10 題 RAG")


### 實作 B2：CILLM GPU Server Vector DB 10 題 RAG

B2 的資料流：

`POST /v1/vector-dbs` → `POST /v1/vector-dbs/{id}/qas` → `POST /v1/vector-dbs/{id}/query` → `LangChain prompt` → `CILLM Chat API`


In [ ]:
required_for_vector_db = {
    "vector_db.manage", "vector_db.upload", "vector_db.query",
    "vector_db.list", "vector_db.read", "vector_db.item_read", "embedding.query",
}

missing = [] if "*" in scopes else sorted(required_for_vector_db - scopes)
if missing:
    print("這支 key 缺少以下 scopes，後續 Vector DB API 可能會失敗：")
    for scope in missing:
        print("-", scope)
else:
    print("Vector DB 練習所需 scopes 看起來足夠。")

STUDENT_ID = os.getenv("CILLM_STUDENT_ID") or input("請輸入你的學員代號，例如 S01：").strip() or "S00"
RUN_ID = os.getenv("CILLM_RUN_ID") or datetime.now().strftime("%m%d%H%M")
CREATED_VECTOR_DB_IDS = []
print("STUDENT_ID:", STUDENT_ID)
print("RUN_ID:", RUN_ID)


In [ ]:
def create_vector_db(name, description, visibility="private"):
    vector_db = api_request("POST", "/v1/vector-dbs", {
        "vector_db_name": name,
        "vector_db_description": description,
        "default_embedding_models": [EMBED_MODEL],
        "visibility": visibility,
    }, timeout=180)
    CREATED_VECTOR_DB_IDS.append(vector_db["vector_db_id"])
    return vector_db


pax_vdb, pax_vdb_create_seconds = timed_call(
    create_vector_db,
    f"{STUDENT_ID}-{RUN_ID}-B-旅客服務與行李FAQ",
    "旅客服務、線上報到、手提與託運行李、特殊餐點、輪椅、寵物託運與航班延誤查詢相關常見問題。",
)
PAX_VECTOR_DB_ID = pax_vdb["vector_db_id"]
print("PAX_VECTOR_DB_ID =", PAX_VECTOR_DB_ID)
print(f"建立 Vector DB 耗時 {pax_vdb_create_seconds:.2f} 秒")
pprint(pax_vdb)

upload_pax, upload_pax_seconds = timed_call(
    api_request,
    "POST",
    f"/v1/vector-dbs/{PAX_VECTOR_DB_ID}/qas",
    {"embedding_models": [EMBED_MODEL], "items": pax_qa},
    timeout=300,
)
print("上傳數量：", upload_pax["total_items"])
print("第一筆 qa_id：", upload_pax["items"][0]["qa_id"])
print(f"上傳與 embedding 耗時 {upload_pax_seconds:.2f} 秒")


In [ ]:
def query_vector_db(vector_db_id, question, top_k=3):
    return api_request("POST", f"/v1/vector-dbs/{vector_db_id}/query", {
        "query": question,
        "embedding_model": EMBED_MODEL,
        "source_types": ["qa"],
        "top_k": {"qa": top_k},
    }, timeout=180)


def format_remote_qa_hits(qa_hits):
    return "\n\n".join(
        f"[排名 {hit['rank']} / 分數 {hit['score']:.4f}]\n問題：{hit['question']}\n答案：{hit['answer']}\n來源：{hit.get('ref')}"
        for hit in qa_hits
    )


def remote_rag_chat(vector_db_id, user_question, top_k=3):
    started = time.perf_counter()
    retrieval, retrieval_seconds = timed_call(query_vector_db, vector_db_id, user_question, top_k=top_k)
    qa_hits = retrieval.get("results", {}).get("qa", []) or []
    context = format_remote_qa_hits(qa_hits)
    prompt_value = rag_prompt.invoke({"context": context, "question": user_question})
    answer, generation_seconds = timed_call(invoke_langchain_text, prompt_value)
    total_seconds = time.perf_counter() - started
    return {
        "question": user_question,
        "answer": answer,
        "retrieval": retrieval,
        "refs": [hit.get("ref") for hit in qa_hits],
        "retrieval_seconds": retrieval_seconds,
        "generation_seconds": generation_seconds,
        "total_seconds": total_seconds,
    }


practice_b2_results = []
for question in pax_benchmark_questions:
    result = remote_rag_chat(PAX_VECTOR_DB_ID, question, top_k=3)
    practice_b2_results.append(result)
    print("\nQ:", question)
    print("refs:", result["refs"])
    print("seconds:", round(result["total_seconds"], 2))
    print(result["answer"])

print_timing_table(practice_b2_results, title="實作 B2：CILLM GPU Server Vector DB 10 題 RAG")


### B1 / B2 時間比較

這格把同一批 10 題的 B1 與 B2 時間放在一起看。B1 的 retriever 在 notebook 記憶體內搜尋；B2 的檢索會走 CILLM GPU Server、Embedding API、Vector DB/Qdrant 與 Gateway。


In [ ]:
comparison_rows = []
for idx, (b1, b2) in enumerate(zip(practice_b1_results, practice_b2_results), 1):
    comparison_rows.append({
        "#": idx,
        "question": b1["question"],
        "b1_seconds": b1["total_seconds"],
        "b2_seconds": b2["total_seconds"],
        "delta_b2_minus_b1": b2["total_seconds"] - b1["total_seconds"],
    })

print(f"{'#':>2}  {'question':<32} {'B1':>8} {'B2':>8} {'B2-B1':>9}")
print("-" * 72)
for row in comparison_rows:
    print(f"{row['#']:>2}  {row['question'][:32]:<32} {row['b1_seconds']:>8.2f} {row['b2_seconds']:>8.2f} {row['delta_b2_minus_b1']:>9.2f}")

b1_total = sum(row["total_seconds"] for row in practice_b1_results)
b2_total = sum(row["total_seconds"] for row in practice_b2_results)
print("\nB1 total:", round(b1_total, 2), "seconds")
print("B2 total:", round(b2_total, 2), "seconds")
print("B2 - B1:", round(b2_total - b1_total, 2), "seconds")
print("B2 / B1:", round(b2_total / b1_total, 2) if b1_total else None)


## 實作 C：另一組 10 題 QA + Pydantic Structured Output 路由

投影片 C 的重點是：當有多個 Vector DB 時，不要只靠人工指定，要先讓 LLM 以結構化輸出判斷應該查哪個知識庫。

這裡新增第二組 10 題 QA，主題刻意和 B 的旅客服務不同：**機務維修與航機放行**。


In [ ]:
maintenance_qa = [
    {"question": "航機放行前維修紀錄需要確認哪些基本項目？", "answer": "放行前需確認工單狀態、缺失處置、必要簽核、適航限制與航機狀態均符合放行條件。", "ref": "CI-MNT-RELEASE-2026#1.1"},
    {"question": "MEL 項目可以無限期延後修復嗎？", "answer": "不可以。MEL 項目需依分類與期限完成修復或展延，逾期不得作為正常放行依據。", "ref": "CI-MNT-MEL-2026#2.4"},
    {"question": "維修人員完成工項後為什麼要做工具清點？", "answer": "工具清點用於避免工具遺留於航機區域，降低異物損傷與飛安風險。", "ref": "CI-MNT-TOOL-2026#3.3"},
    {"question": "航機外觀檢查發現雷擊痕跡時應如何處理？", "answer": "應依雷擊檢查程序記錄位置、評估損傷範圍，必要時執行進一步檢查並完成簽核後才能放行。", "ref": "CI-MNT-LIGHTNING-2026#4.2"},
    {"question": "重複性故障應如何判斷是否需要升級處理？", "answer": "若同一系統或同類故障在指定期間內重複發生，需依可靠度與維修管制程序升級分析。", "ref": "CI-MNT-REPEAT-2026#2.2"},
    {"question": "航機加油後需要確認哪些紀錄？", "answer": "需確認加油量、油品等級、油單、油量平衡與相關簽認紀錄。", "ref": "CI-MNT-FUEL-2026#1.5"},
    {"question": "維修交接紀錄的目的為何？", "answer": "交接紀錄用於確保未完成工項、風險、限制與後續處置能被下一班組完整掌握。", "ref": "CI-MNT-HANDOVER-2026#5.1"},
    {"question": "航機放行簽署者需要具備什麼條件？", "answer": "放行簽署者需具備相應授權、機型資格與有效訓練紀錄，並確認相關維修資料完整。", "ref": "CI-MNT-AUTH-2026#2.1"},
    {"question": "發現航材標籤資訊不完整時可以直接使用嗎？", "answer": "不可以。航材標籤、料號、序號或適航文件不完整時，需先隔離並釐清來源與適用性。", "ref": "CI-MNT-PART-2026#3.6"},
    {"question": "維修後功能測試失敗時可以用口頭說明放行嗎？", "answer": "不可以。功能測試失敗需依程序排故、記錄結果並完成必要簽核，不能僅以口頭說明替代。", "ref": "CI-MNT-FUNC-2026#4.4"},
]

maintenance_benchmark_questions = [
    "航機放行前要先確認哪些維修紀錄？",
    "MEL 缺失可以一直延後修復嗎？",
    "完成維修後為什麼還要做工具清點？",
    "外觀檢查看到疑似雷擊痕跡時要怎麼辦？",
    "同一個系統一直故障，什麼時候要升級分析？",
    "航機加油後需要核對哪些資料？",
    "維修交接紀錄主要是為了什麼？",
    "誰可以簽署航機放行？",
    "航材標籤資訊不完整時能不能先使用？",
    "功能測試失敗可以口頭說明後放行嗎？",
]

print("第二組 QA 數量：", len(maintenance_qa))
print("第二組 benchmark 問題數量：", len(maintenance_benchmark_questions))


In [ ]:
mnt_vdb, mnt_vdb_create_seconds = timed_call(
    create_vector_db,
    f"{STUDENT_ID}-{RUN_ID}-C-機務維修與航機放行FAQ",
    "機務維修、航機放行、MEL、工具清點、雷擊檢查、維修交接、航材標籤與功能測試相關常見問題。",
)
MNT_VECTOR_DB_ID = mnt_vdb["vector_db_id"]
print("MNT_VECTOR_DB_ID =", MNT_VECTOR_DB_ID)
print(f"建立 Vector DB 耗時 {mnt_vdb_create_seconds:.2f} 秒")

upload_mnt, upload_mnt_seconds = timed_call(
    api_request,
    "POST",
    f"/v1/vector-dbs/{MNT_VECTOR_DB_ID}/qas",
    {"embedding_models": [EMBED_MODEL], "items": maintenance_qa},
    timeout=300,
)
print("上傳數量：", upload_mnt["total_items"])
print(f"上傳與 embedding 耗時 {upload_mnt_seconds:.2f} 秒")


In [ ]:
class VectorDbRoute(BaseModel):
    choose: Literal["A", "B"] = Field(description="A 代表旅客服務與行李；B 代表機務維修與航機放行")
    vector_db_id: str = Field(description="候選清單中的 vector_db_id")
    reason: str = Field(description="選擇這個 vector DB 的原因")
    confidence: float = Field(ge=0, le=1, description="0 到 1 之間的信心分數")


candidate_vector_dbs = [
    {
        "choose": "A",
        "vector_db_id": PAX_VECTOR_DB_ID,
        "name": pax_vdb["vector_db_name"],
        "description": pax_vdb["vector_db_description"],
    },
    {
        "choose": "B",
        "vector_db_id": MNT_VECTOR_DB_ID,
        "name": mnt_vdb["vector_db_name"],
        "description": mnt_vdb["vector_db_description"],
    },
]

route_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
你是知識庫路由器。請根據使用者問題與 vector DB description，選出最適合的 vector DB。
你必須輸出合法 JSON，不要輸出 Markdown，不要輸出解釋文字。
JSON 欄位必須包含 choose, vector_db_id, reason, confidence。
choose 只能是 A 或 B；confidence 必須是 0 到 1 的數字。
""".strip(),
    ),
    (
        "human",
        """
候選 vector DB：
{candidates}

使用者問題：
{question}
""".strip(),
    ),
])

structured_route_llm = CILLM_LANGCHAIN_MODEL.with_structured_output(
    VectorDbRoute,
    method="json_mode",
)


def extract_json_object(text):
    left = text.find("{")
    right = text.rfind("}")
    if left == -1 or right == -1 or right <= left:
        raise ValueError(f"找不到 JSON object：{text}")
    return text[left:right + 1]


def choose_vector_db(user_question, candidates):
    prompt_value = route_prompt.invoke({
        "candidates": json.dumps(candidates, ensure_ascii=False, indent=2),
        "question": user_question,
    })

    try:
        route = structured_route_llm.invoke(prompt_value.to_messages())
    except Exception as exc:
        print("LangChain structured output 解析失敗，改用 CILLM JSON mode fallback：", repr(exc))
        raw = chat(
            prompt_value_to_cillm_messages(prompt_value),
            model=DEFAULT_CHAT_MODEL,
            temperature=0,
            max_tokens=300,
            response_format={"type": "json_object"},
        )
        route = VectorDbRoute.model_validate_json(extract_json_object(raw))

    allowed_by_id = {item["vector_db_id"] for item in candidates}
    allowed_by_label = {item["choose"]: item["vector_db_id"] for item in candidates}

    if route.vector_db_id not in allowed_by_id:
        repaired_id = allowed_by_label.get(route.choose)
        if repaired_id:
            route = route.model_copy(update={"vector_db_id": repaired_id})
        else:
            raise ValueError(f"LLM 選到不存在的 vector_db_id：{route.model_dump()}")
    return route


route = choose_vector_db("航機雷擊後放行前要注意什麼？", candidate_vector_dbs)
pprint(route.model_dump())


In [ ]:
def smart_rag_chat(user_question, top_k=3):
    started = time.perf_counter()
    route, routing_seconds = timed_call(choose_vector_db, user_question, candidate_vector_dbs)
    rag_result = remote_rag_chat(route.vector_db_id, user_question, top_k=top_k)
    total_seconds = time.perf_counter() - started
    return {
        "question": user_question,
        "route": route,
        "answer": rag_result["answer"],
        "retrieval": rag_result["retrieval"],
        "refs": rag_result["refs"],
        "routing_seconds": routing_seconds,
        "retrieval_seconds": rag_result["retrieval_seconds"],
        "generation_seconds": rag_result["generation_seconds"],
        "total_seconds": total_seconds,
    }


practice_c_results = []
for question in maintenance_benchmark_questions:
    result = smart_rag_chat(question, top_k=3)
    practice_c_results.append(result)
    print("\nQ:", question)
    print("route:", result["route"].model_dump())
    print("refs:", result["refs"])
    print("seconds:", round(result["total_seconds"], 2))
    print(result["answer"])

print_timing_table(practice_c_results, title="實作 C：Structured Output Router + CILLM Vector DB 10 題 RAG")


## 課後檢查：列出目前建立的 Vector DB

這格可以確認剛剛 B2/C 建立的 Vector DB 是否存在，也能看每個庫的 embedding counts。


In [ ]:
all_vector_dbs = api_request("GET", "/v1/vector-dbs")
for vector_db in all_vector_dbs:
    if vector_db["vector_db_id"] in CREATED_VECTOR_DB_IDS:
        print(vector_db["vector_db_id"], vector_db["vector_db_name"], vector_db["visibility"], vector_db["enabled"])
        for model in vector_db.get("embedding_models", []):
            print("  -", model["embedding_model"], "qa=", model["qa_embedding_count"], "chunk=", model["chunk_embedding_count"])